# Controlled signal discovery results

This notebook compares full-list rankers (Random, MiniLM, Qwen3-Reranker, and Decider) with two Decider graph-search variants: model-selected root and oracle-known root. Both rerank the discovered shortlist using the intent plus graph-search history. The candidate catalog is the standard 39-signal RelAmazon schema-only pool. Any relevant signal not discovered counts as not retrieved; the ideal nDCG and recall denominators still use all qrels. The builder copies schema metadata only, not source records. This notebook reads saved results; it does not rerun models.

The suite covers direct attributes, one- and two-hop paths, relationship-only signals, multiple relevant signals, distractors, AND/OR logic, and temporal cases. Grades are author-defined synthetic qrels—not human-validated judgments—so this is a controlled capability check, not evidence of performance on real user conditions. The two nDCG heatmaps show path-exact scores and, as a diagnostic only, provenance-soft scores that collapse alternate paths to the same underlying attribute or relationship. The final Decider prompt receives the user intent and a summary of graph-search history, not the full schema. Its probabilities are conditional on the discovered shortlist; undiscovered candidates remain unretrieved.

In [ ]:
from pathlib import Path
import json
import os
MPL_CONFIG = Path('/tmp/simbench-matplotlib')
MPL_CONFIG.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(MPL_CONFIG))
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

def plot_with_oracle_root_delta(frame, method_col, metrics, title, ylabel, figsize, metric_axis=False):
    # Keep oracle-root as a hatched extension of the model-root graph bar, not a separate method.
    from matplotlib.patches import Patch
    graph_method = 'decider-graph'
    oracle_method = 'decider-graph-oracle-root'
    labels = [str(value) for value in frame[method_col].dropna().tolist()]
    methods = list(dict.fromkeys(label for label in labels if label != oracle_method))
    graph_rows = frame.loc[frame[method_col].astype(str) == graph_method]
    oracle_rows = frame.loc[frame[method_col].astype(str) == oracle_method]
    graph_values = graph_rows.iloc[0] if not graph_rows.empty else None
    oracle_values = oracle_rows.iloc[0] if not oracle_rows.empty else None
    fig, ax = plt.subplots(figsize=figsize)
    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
    delta_handles = []
    if metric_axis:
        positions = list(range(len(metrics)))
        width = 0.8 / max(1, len(methods))
        for method_index, method in enumerate(methods):
            rows = frame.loc[frame[method_col].astype(str) == method]
            if rows.empty:
                continue
            row = rows.iloc[0]
            values = [float(row[metric]) for metric in metrics]
            offset = (method_index - (len(methods) - 1) / 2) * width
            color = colors[method_index % len(colors)]
            ax.bar([position + offset for position in positions], values, width, label=method, color=color)
            if method == graph_method and oracle_values is not None:
                for position, base, oracle in zip(positions, values, [float(oracle_values[metric]) for metric in metrics]):
                    ax.bar(position + offset, abs(oracle - base), width, bottom=min(base, oracle), facecolor='none', edgecolor=color, hatch='....', linewidth=1.2, zorder=4)
        ax.set_xticks(positions, metrics)
        ax.legend(title='Method', bbox_to_anchor=(1.02, 1), loc='upper left')
    else:
        positions = list(range(len(methods)))
        width = 0.8 / max(1, len(metrics))
        for metric_index, metric in enumerate(metrics):
            values = [float(frame.loc[frame[method_col].astype(str) == method, metric].iloc[0]) for method in methods]
            offset = (metric_index - (len(metrics) - 1) / 2) * width
            color = colors[metric_index % len(colors)]
            ax.bar([position + offset for position in positions], values, width, label=metric, color=color)
            if graph_method in methods and oracle_values is not None:
                graph_index = methods.index(graph_method)
                base = values[graph_index]
                oracle = float(oracle_values[metric])
                ax.bar(positions[graph_index] + offset, abs(oracle - base), width, bottom=min(base, oracle), facecolor='none', edgecolor=color, hatch='....', linewidth=1.2, zorder=4)
        ax.set_xticks(positions, methods)
        ax.legend(title='Metric', bbox_to_anchor=(1.02, 1), loc='upper left')
        plt.xticks(rotation=25, ha='right')
    if oracle_values is not None and graph_values is not None:
        delta_handles.append(Patch(facecolor='white', edgecolor='0.25', hatch='....', label='Oracle-root delta'))
        handles, legend_labels = ax.get_legend_handles_labels()
        ax.legend(handles + delta_handles, legend_labels + ['Oracle-root delta'], title='Method' if metric_axis else 'Metric', bbox_to_anchor=(1.02, 1), loc='upper left')
    ax.set_ylim(0, 1.05)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(axis='y', alpha=0.25)
    plt.tight_layout()
    plt.show()

RESULT_DIR = Path('/storage/lcostes/ray-share/experiments/simbench-signals/controlled-rel-amazon-v2-ray-20261012')
SUMMARY_PATH = RESULT_DIR / 'summary.json'
TASKS_PATH = RESULT_DIR / 'summary-tasks.jsonl'
GRAPH_RERANKED_SUMMARY_PATH = RESULT_DIR / 'decider-graph-reranked-summary.json'
GRAPH_RERANKED_TASKS_PATH = RESULT_DIR / 'decider-graph-reranked-summary-tasks.jsonl'
ORACLE_ROOT_SUMMARY_PATH = RESULT_DIR / 'decider-graph-reranked-oracle-root-summary.json'
ORACLE_ROOT_TASKS_PATH = RESULT_DIR / 'decider-graph-reranked-oracle-root-summary-tasks.jsonl'

if not SUMMARY_PATH.is_file():
    raise FileNotFoundError(f'Missing aggregate results: {SUMMARY_PATH}')
if not TASKS_PATH.is_file():
    raise FileNotFoundError(f'Missing per-task results: {TASKS_PATH}')

summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
task_rows = [json.loads(line) for line in TASKS_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
if GRAPH_RERANKED_SUMMARY_PATH.is_file() and GRAPH_RERANKED_TASKS_PATH.is_file():
    graph_reranked_summary = json.loads(GRAPH_RERANKED_SUMMARY_PATH.read_text(encoding='utf-8'))
    if graph_reranked_summary.get('search_config', {}).get('root_branch_policy') != 'branch_width_top_actions_at_root':
        print('WARNING: saved model-root graph results predate root branching; rerun the model-root evaluator before interpreting them.')
    graph_reranked_rows = [json.loads(line) for line in GRAPH_RERANKED_TASKS_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
    summary['scorers'].update(graph_reranked_summary['scorers'])
    task_rows.extend(graph_reranked_rows)
    print(f"Loaded graph-search + Decider shortlist-reranking results from {GRAPH_RERANKED_SUMMARY_PATH}")
else:
    print(f"Graph shortlist reranking not yet evaluated; expected {GRAPH_RERANKED_SUMMARY_PATH}")
if ORACLE_ROOT_SUMMARY_PATH.is_file() and ORACLE_ROOT_TASKS_PATH.is_file():
    oracle_root_summary = json.loads(ORACLE_ROOT_SUMMARY_PATH.read_text(encoding='utf-8'))
    oracle_root_rows = [json.loads(line) for line in ORACLE_ROOT_TASKS_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
    summary['scorers'].update(oracle_root_summary['scorers'])
    task_rows.extend(oracle_root_rows)
    print(f"Loaded oracle-root graph-search + history-reranking results from {ORACLE_ROOT_SUMMARY_PATH}")
else:
    print(f"Oracle-root graph reranking not yet evaluated; expected {ORACLE_ROOT_SUMMARY_PATH}")
task_count = len({row['task_id'] for row in task_rows})
METHOD_ORDER = ['random', 'minilm', 'qwen3-reranker', 'decider', 'decider-graph-reranked', 'decider-graph-reranked-oracle-root']
METHOD_LABELS = {'decider-graph-reranked': 'decider-graph', 'decider-graph-reranked-oracle-root': 'decider-graph-oracle-root'}
scorer_names = [name for name in METHOD_ORDER if any(row['scorer'] == name for row in task_rows)]
summary['scorers'] = {name: summary['scorers'][name] for name in scorer_names if name in summary['scorers']}
print(f"Benchmark: {summary['benchmark']} | tasks: {task_count} | scorers: {scorer_names} | rows: {len(task_rows)} | files: {RESULT_DIR}")
assert task_count == 25, f'Expected 25 tasks, found {task_count}'
assert all(row['candidate_count'] == 39 for row in task_rows), 'Expected the standard 39-candidate pool in every task'
required_scorers = {'random', 'minilm', 'qwen3-reranker', 'decider'}
assert required_scorers <= set(scorer_names), f'Missing existing baseline scorer(s): {required_scorers - set(scorer_names)}'
assert set(scorer_names) <= required_scorers | {'decider-graph-reranked', 'decider-graph-reranked-oracle-root'}

In [ ]:
# Recompute paired path-exact and provenance-soft metrics from the saved top-five rankings.
# Soft relevance collapses path variants of one physical attribute/FK relation; duplicate cores count once.
import sys
sys.path.insert(0, '/storage/lcostes/Similarity_benchmark')
from src.synthetic.controlled_signal_metrics import ranking_metrics
BENCHMARK_DIR = Path('/storage/lcostes/ray-share/experiments/simbench-signals/controlled-rel-amazon-signals-v2')
candidate_rows = json.loads((BENCHMARK_DIR / 'candidate_catalog.json').read_text(encoding='utf-8'))
candidate_metadata = {row['id']: row for row in candidate_rows}
task_qrels = {row['id']: row['signal_grades'] for row in (json.loads(line) for line in (BENCHMARK_DIR / 'tasks.jsonl').read_text(encoding='utf-8').splitlines() if line.strip())}
for row in task_rows:
    row.update(ranking_metrics([signal['id'] for signal in row['top_signals']], task_qrels[row['task_id']], candidate_metadata))
dual_metrics = [c for c in task_rows[0] if c.startswith(('path_exact_', 'provenance_soft_'))]
for scorer in summary['scorers']:
    subset = [row for row in task_rows if row['scorer'] == scorer]
    for metric in dual_metrics:
        summary['scorers'][scorer][metric] = sum(float(row[metric]) for row in subset) / len(subset)

# Aggregate comparison, macro-averaged equally over all 25 conditions.
aggregate = pd.DataFrame.from_dict(summary['scorers'], orient='index').rename_axis('scorer').reset_index()
aggregate['scorer'] = aggregate['scorer'].map(METHOD_LABELS).fillna(aggregate['scorer'])
metric_columns = ['ndcg@1', 'ndcg@3', 'ndcg@5', 'recall@1', 'recall@3', 'recall@5', 'mrr']
display(aggregate[['scorer', 'task_count'] + metric_columns].style.format({c: '{:.3f}' for c in metric_columns}))

In [ ]:
# Aggregate metrics. Oracle-root is shown as a dotted delta on the model-root graph bars.
plot_metrics = ['ndcg@1', 'ndcg@3', 'ndcg@5', 'recall@1', 'recall@3', 'recall@5']
plot_with_oracle_root_delta(aggregate, 'scorer', plot_metrics, 'Controlled signal-discovery ranking quality', 'Macro average across conditions', (11, 5), metric_axis=True)

In [ ]:
# Per-condition nDCG@3 view: strict path identity and provenance-soft credit.
per_task = pd.DataFrame(task_rows)
task_order = list(dict.fromkeys(per_task['task_id']))
scorer_order = [name for name in METHOD_ORDER if name in summary['scorers']]
heat = per_task.pivot(index='scorer', columns='task_id', values='provenance_soft_ndcg@3').reindex(index=scorer_order, columns=task_order)
fig, ax = plt.subplots(figsize=(20, 7))
image = ax.imshow(heat.to_numpy(), vmin=0, vmax=1, cmap='viridis', aspect='auto')
short_labels = [task.replace('-', '\n') for task in task_order]
ax.set_xticks(range(len(task_order)), short_labels, fontsize=8)
ax.set_yticks(range(len(scorer_order)), [METHOD_LABELS.get(name, name) for name in scorer_order])
ax.set_title('Provenance-soft nDCG@3 by method and synthetic condition')
for i in range(len(scorer_order)):
    for j in range(len(task_order)):
        ax.text(j, i, f'{heat.iloc[i, j]:.2f}', ha='center', va='center', color='white' if heat.iloc[i, j] < 0.55 else 'black')
fig.colorbar(image, ax=ax, label='nDCG@3')
plt.tight_layout()
plt.show()

In [ ]:
# Matching path-exact heatmap, using the same scale as the provenance-soft view above.
heat_exact = per_task.pivot(index='scorer', columns='task_id', values='path_exact_ndcg@3').reindex(index=scorer_order, columns=task_order)
fig, ax = plt.subplots(figsize=(20, 7))
image = ax.imshow(heat_exact.to_numpy(), vmin=0, vmax=1, cmap='viridis', aspect='auto')
ax.set_xticks(range(len(task_order)), short_labels, fontsize=8)
ax.set_yticks(range(len(scorer_order)), [METHOD_LABELS.get(name, name) for name in scorer_order])
ax.set_title('Path-exact nDCG@3 by method and synthetic condition')
for i in range(len(scorer_order)):
    for j in range(len(task_order)):
        ax.text(j, i, f'{heat_exact.iloc[i, j]:.2f}', ha='center', va='center', color='white' if heat_exact.iloc[i, j] < 0.55 else 'black')
fig.colorbar(image, ax=ax, label='nDCG@3')
plt.tight_layout(); plt.show()

In [ ]:
# Browse the per-task method comparison.
def show_task(task_id):
    selected = per_task[per_task['task_id'] == task_id].copy()
    if selected.empty:
        raise KeyError(f'No such task: {task_id}; available: {task_order}')
    print(f"{selected.iloc[0]['reasoning_type']} | {selected.iloc[0]['condition']}")

    top_rows = []
    for _, row in selected.iterrows():
        for signal in row['top_signals']:
            top_rows.append({
                'method': row['scorer'],
                'reasoning type': row['reasoning_type'],
                'rank': signal['rank'],
                'grade': signal['grade'],
                'signal': signal['description'],
                'nDCG@3': row['ndcg@3'],
            })
    top_table = pd.DataFrame(top_rows)
    top_table['method'] = top_table['method'].map(METHOD_LABELS).fillna(top_table['method'])
    top_table['method'] = pd.Categorical(top_table['method'], categories=[METHOD_LABELS.get(name, name) for name in METHOD_ORDER], ordered=True)
    display(top_table[top_table['rank'] <= 5].sort_values(['method', 'rank']).style.format({'nDCG@3': '{:.3f}'}))

widgets.interact(show_task, task_id=widgets.Dropdown(options=task_order, description='Task'))

In [ ]:
# Compact per-condition metric table. Useful for copying into notes.
if 'returned_signal_count' not in per_task:
    per_task['returned_signal_count'] = per_task['candidate_count']
else:
    per_task['returned_signal_count'] = per_task['returned_signal_count'].fillna(per_task['candidate_count'])
condition_metrics = per_task[['scorer', 'reasoning_type', 'task_id', 'candidate_count', 'returned_signal_count', 'ndcg@3', 'recall@3', 'mrr']].copy()
condition_metrics['scorer'] = condition_metrics['scorer'].map(METHOD_LABELS).fillna(condition_metrics['scorer'])
condition_metrics['scorer'] = pd.Categorical(condition_metrics['scorer'], categories=[METHOD_LABELS.get(name, name) for name in METHOD_ORDER], ordered=True)
condition_metrics = condition_metrics.sort_values(['reasoning_type', 'task_id', 'scorer'])
display(condition_metrics.style.format({'ndcg@3': '{:.3f}', 'recall@3': '{:.3f}', 'mrr': '{:.3f}'}))

## Reading these results

The paired heatmaps compare path-exact relevance with a provenance-soft diagnostic that treats alternate routes to the same physical attribute or FK relation as equivalent. For graph-search, only discovered signals are ranked; undiscovered signals remain misses.

There are 25 curated synthetic conditions, but some reasoning categories have only one task, so per-category conclusions are not statistically robust. Inspect the heatmap and top-signal table alongside the macro average. The qrels encode the suite author's intended mapping; they do not establish which signals people prefer for real RelAmazon conditions.

## Decider graph search with history-conditioned final ranking

The graph procedure first discovers a bounded shortlist. A final Decider choice then orders only that shortlist, using the intent and the recorded exploration history rather than the full schema. Compare its per-condition results with the independent full-list Decider baseline below. The score still penalizes relevant qrel signals that graph search never discovered.

In [ ]:
method_summary = pd.DataFrame.from_dict(summary['scorers'], orient='index').rename_axis('method').reset_index()
method_summary['method'] = method_summary['method'].map(METHOD_LABELS).fillna(method_summary['method'])
display(method_summary[['method', 'task_count', 'ndcg@3', 'recall@3', 'mrr']].style.format({'ndcg@3': '{:.3f}', 'recall@3': '{:.3f}', 'mrr': '{:.3f}'}))
plot_with_oracle_root_delta(method_summary, 'method', ['ndcg@3', 'recall@3', 'mrr'], 'Full-list baselines versus graph discovery + history-conditioned reranking', 'Macro average over controlled conditions', (11, 4.5))

## Per-condition comparison: full-list Decider and both graph variants

The oracle-root variant is an ablation: it receives the controlled task's authored source table, while normal graph search retains up to `branch_width` high-probability root tables and explores their branches under the same total model-call budget. Both use the same history-conditioned final reranker. Scores use the full qrels, so missed signals remain misses. Model-root results report top-choice accuracy and whether the authored source table is included among explored roots.

In [ ]:
required_graph_methods = {'decider-graph-reranked', 'decider-graph-reranked-oracle-root'}
if not required_graph_methods <= set(per_task['scorer']):
    print(f'Run both graph + history-rerank evaluations first; expected {GRAPH_RERANKED_SUMMARY_PATH} and {ORACLE_ROOT_SUMMARY_PATH}')
else:
    model_root_rows = per_task[per_task['scorer'] == 'decider-graph-reranked']
    if len(model_root_rows) and isinstance(model_root_rows.iloc[0].get('root_selection'), dict):
        root_records = pd.DataFrame([{'task_id': row['task_id'], **row['root_selection']} for _, row in model_root_rows.iterrows()])
        root_accuracy = root_records['correct'].astype(bool).mean()
        print(f'Model-selected root accuracy: {root_records["correct"].astype(bool).sum()}/{len(root_records)} = {root_accuracy:.1%} against the task-defined source_table.')
        if 'explored_tables' in root_records:
            root_records['source_in_explored_roots'] = root_records.apply(lambda r: r['expected_table'] in r['explored_tables'], axis=1)
            hit_rate = root_records['source_in_explored_roots'].astype(bool).mean()
            print(f'Authored source table included among explored roots: {root_records["source_in_explored_roots"].astype(bool).sum()}/{len(root_records)} = {hit_rate:.1%}.')
        display(root_records[[c for c in ['task_id', 'expected_table', 'selected_table', 'explored_tables', 'choice_probability', 'correct'] if c in root_records]].style.format({'choice_probability': '{:.3f}'}))
    else:
        print('Root-choice accuracy is not present in these saved results. Rerun the model-root evaluator with the current code to save root choices and probabilities.')
    graph_rows = per_task[per_task['scorer'].isin(['decider-graph-reranked', 'decider-graph-reranked-oracle-root'])]
    score_pair = graph_rows.pivot(index='task_id', columns='scorer', values='path_exact_ndcg@3').dropna()
    mrr_pair = graph_rows.pivot(index='task_id', columns='scorer', values='mrr').reindex(score_pair.index)
    paired = pd.DataFrame({'model_root_ndcg@3': score_pair['decider-graph-reranked'], 'oracle_root_ndcg@3': score_pair['decider-graph-reranked-oracle-root'], 'delta_ndcg@3': score_pair['decider-graph-reranked-oracle-root'] - score_pair['decider-graph-reranked'], 'delta_mrr': mrr_pair['decider-graph-reranked-oracle-root'] - mrr_pair['decider-graph-reranked']})
    improved = int((paired['delta_ndcg@3'] > 1e-12).sum())
    unchanged = int((paired['delta_ndcg@3'].abs() <= 1e-12).sum())
    regressed = int((paired['delta_ndcg@3'] < -1e-12).sum())
    print(f"Oracle-root effect: mean paired ΔnDCG@3={paired['delta_ndcg@3'].mean():+.3f}; improved {improved}/{len(paired)}, unchanged {unchanged}/{len(paired)}, regressed {regressed}/{len(paired)}. Mean paired ΔMRR={paired['delta_mrr'].mean():+.3f}.")
    display(paired.sort_values('delta_ndcg@3', ascending=False).style.format('{:+.3f}'))
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].scatter(paired['model_root_ndcg@3'], paired['oracle_root_ndcg@3'], c=paired['delta_ndcg@3'], cmap='coolwarm', vmin=-1, vmax=1, s=55, edgecolor='white')
    axes[0].plot([0, 1], [0, 1], '--', color='gray', linewidth=1)
    axes[0].set(xlim=(-0.03, 1.03), ylim=(-0.03, 1.03), xlabel='Model-selected root nDCG@3', ylabel='Oracle-root nDCG@3', title='Paired task scores')
    axes[0].grid(alpha=0.25)
    ordered_delta = paired['delta_ndcg@3'].sort_values()
    axes[1].barh(ordered_delta.index, ordered_delta.values, color=['#d95f5f' if value < 0 else '#36a269' for value in ordered_delta.values])
    axes[1].axvline(0, color='black', linewidth=1)
    axes[1].set(xlabel='Oracle-root minus model-root nDCG@3', ylabel='Synthetic condition', title='Per-condition root effect')
    axes[1].grid(axis='x', alpha=0.25)
    plt.tight_layout()
    plt.show()